# Mushroom body segmentation via hand-labeled SAM prompts (2026_07_30_Lisa)

New strategy, replacing the atlas-registration + adaptive-threshold pipeline from
`26_07_30_Lisa_MB_registration_parts.ipynb` -- that approach's masks were unreliable enough
(disconnected blobs, inconsistent tapering) that it's not worth continuing to patch. Instead:

1. Hand-label the MB on 2-3 representative Z-slices, interactively, using micro-sam's own
   `annotator_3d` tool -- you click point/box prompts, SAM proposes a mask live, you correct
   and commit it per slice.
2. Use the tool's built-in volumetric propagation to extend those committed slices through
   the rest of the stack.

This is micro-sam's actual designed use case (interactive annotation + propagation), unlike
the previous notebook's ad-hoc use of a registration-derived mask as an automatic prompt for
every frame independently.

In [1]:
import os
os.environ['QT_QPA_PLATFORM'] = 'xcb'

In [ ]:
import sys
import platform
import numpy as np
from tifffile import imread

system = platform.system()
if system == 'Linux':
    home = '/home/gerard/'
elif system == 'Darwin':
    home = '/Users/gerard/'
elif system == 'Windows':
    home = 'C:/Users/cviko/'

data_home = home + 'data/confocal/'


## 1. Load one scene's nc82 channel

This notebook runs in `micro-sam-env`, which deliberately has no `aicsimageio` installed (see
the OpenGL-crash investigation above/in project notes -- `micro-sam-env` needs a newer
`tifffile`/`napari`/`vispy` combination than `aicsimageio` supports). So the `.lif` is never
touched here directly. Instead, run `lif_saver.ipynb` (in `leica-env`) once first for whichever
scene/channel you need -- it loads via `aicsimageio` and writes a plain calibrated TIFF -- and
this cell just reads that TIFF back with bare `tifffile`.

In [ ]:
date = '2026_07_30'
user = 'Lisa'
scene = 0
channel = 0  # ch0 = nc82

# written by lif_saver.ipynb -- run that first (in leica-env) if this file doesn't exist yet
stack_path = data_home + date + '_' + user + f'/series_{scene}/{date}_s{scene}_ch{channel}.tif'

nc82_stack = imread(stack_path).astype(np.float32)
print('nc82 stack shape (ZYX):', nc82_stack.shape)


In [4]:
from qtpy.QtWidgets import QApplication
import os
app = QApplication.instance()
print('QT_QPA_PLATFORM env var:', os.environ.get('QT_QPA_PLATFORM'))
print('Qt platformName actually in use:', app.platformName() if app else 'no QApplication exists yet')


QT_QPA_PLATFORM env var: xcb
Qt platformName actually in use: no QApplication exists yet


## 2. Launch the interactive 3D annotator

`embedding_path` doubles as a cache -- first launch computes and saves embeddings for the
whole stack there (the slow part, ~2.9s/frame on this machine's MPS backend, so a few minutes
for the full 86-frame stack); every later launch (even after a kernel/notebook restart) loads
the cached embeddings from disk instead of recomputing them.

**Once the napari window opens:**
1. Navigate to a representative Z-slice (the slider at the bottom).
2. Use the micro-sam widget panel to click point prompts inside the MB (positive) and,
   if needed, outside it (negative) -- SAM proposes a mask live as you click.
3. Once the proposed mask on that slice looks right, commit it (the widget has a commit
   control -- exact label may vary by micro-sam version, look for it in the panel).
4. Repeat on 2-3 total slices, spread across the stack rather than clustered together.
5. Use the tool's volumetric-segmentation control to propagate the committed slices through
   the rest of the stack.
6. Leave the viewer open and run the next cell to pull out the result once you're satisfied.

In [ ]:
import micro_sam.sam_annotator as sam_annotator

embedding_path = data_home + date + '_' + user + f'/series_{scene}/sam_embeddings.zarr'
os.makedirs(os.path.dirname(embedding_path), exist_ok=True)

annotator_viewer = sam_annotator.annotator_3d(
    nc82_stack,
    embedding_path=embedding_path,
    model_type='vit_b_lm',  # micro-sam's light-microscopy-finetuned checkpoint
    return_viewer=True,
)


/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
progress: : 0it [00:00, ?it/s]
02-Sep-26 15:29:32 - npe2.manifest.schema - ERROR    - napari.manifest -> 'napari' could not be imported: `PluginManifest` is not fully defined; you should define `Draft06JsonSchema`, then call `PluginManifest.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.11/u/class-not-fully-defined
02-Sep-26 15:29:32 - npe2.manifest.schema - ERROR    - napari.manifest -> 'napari-svg' could not be imported: `PluginManifest` is not fully defined; you should define `Draft06JsonSchema`, then call `PluginManifest.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.11/u/class-not-fully-defined
02-Sep-26 15:29:32 - npe2.manifest.schema - ERROR    - n

Error: Attempt to retrieve context when no valid context

02-Sep-26 15:29:34 - vispy    - WARNING  - Error drawing visual <Image at 0x7f9c48ee0e10>
Traceback (most recent call last):
  File "/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/vispy/app/backends/_qt.py", line 1000, in paintGL
    self._vispy_canvas.events.draw(region=None)
  File "/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/vispy/util/event.py", line 471, in _invoke_callback
    _handle_exception(self.ignore_callback_errors,
  File "/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/vispy/util/event.py", line 469, in _invoke_callback
    cb(event)
  File "/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/vispy/scene/canvas.py", line 226, in on_draw
    self._draw_scene()
  File "/home/gerard/miniconda3/envs/leica-env/lib/python3.11/site-packages/vis

## 3. Retrieve the final segmentation

The annotator keeps its result in a `committed_objects` layer on the same viewer -- run this
once you're done annotating/propagating in the napari window above (don't close it first).

In [ ]:
mb_mask_handlabeled = annotator_viewer.layers['committed_objects'].data
mb_mask_handlabeled = mb_mask_handlabeled.astype(bool)

print(f'hand-labeled MB mask: {mb_mask_handlabeled.sum()} voxels '
      f'({100 * mb_mask_handlabeled.sum() / mb_mask_handlabeled.size:.2f}% of volume)')

voxels_per_z = mb_mask_handlabeled.sum(axis=(1, 2))
print('voxel count per Z frame:', voxels_per_z)
